Проверка работы NewareNDA для целей работы библиотеки

# Конфигурация окружения

## Нужные импорты

In [1]:
import logging
import re
import tempfile
import xml.etree.ElementTree as ET
import zipfile
from collections.abc import ValuesView
from copy import copy
from datetime import datetime

import NewareNDA
import pandas as pd
from beartype import beartype

import battery_parser

In [2]:
logger = logging.getLogger('newarenda')

In [3]:
#Параметры окружения
xml_file = r'D:\!Science\Analysis\Electrochem\Test data\NDAX\With barcode and remarks\TestInfo.xml'
xml_settings = r'D:\!Science\Analysis\Electrochem\Test data\XML\simple.xml'
file_ndax = r'D:\!Science\Analysis\Electrochem\Test data\NDAX\With barcode and remarks.ndax'
sa_ndax = r'D:\!Science\Analysis\Electrochem\Test data\NDAX\104-1-8-SA28.ndax'
sa_ndax2 = r'D:\!Science\Analysis\Electrochem\Test data\NDAX\104-1-2-SA07.ndax'
directory = r'D:\!Science\Analysis\Electrochem\2024 Na-ion'
filtered_datapath = directory + '\\' + 'filtered_data.parquet'
step_datapath = directory + '\\' + 'parsed_data.parquet'

### Установка необходимых пакетов

In [ ]:
%pip install NewareNDA
%pip install ipykernel
%pip install pandas
%pip install lxml
%pip install zipfile
%pip install io
%pip install xml.etree.ElementTree
%pip install tempfile
%pip install logging
%pip install beartype
%pip install pyarrow fastparquet

## Проверка состояния окружения

In [ ]:
!python --version
%pip freeze

# Испытания

In [ ]:
file = r"D:\\!Science\\Analysis\\Electrochem\\2024 Na-ion\\2025-01-10 target SoH cycling\\Данные 2025-05-30\\102-1-6-SB46.ndax"
imp = NewareNDA.read(file)

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    zf = zipfile.PyZipFile(file)

    # Read version information
    version_info = zf.extract('TestInfo.xml', path=tmpdir)

    with open(version_info, 'r', encoding='gb2312') as f:
        text = f.read()
        config = ET.fromstring(text)



In [ ]:
@beartype
def parse_et_to_dict(element: ET.Element, element_path: str = '') -> dict[str, dict[str, str]]:
    """Build a mapping from full XML paths to element attribute dicts.

    Parameters
    ----------
    element : xml.etree.ElementTree.Element
        Root element to traverse.
    element_path : str, optional
        Prefix path to start from. Used internally during recursion.

    Returns
    -------
    dict[str, dict[str, str]]
        Mapping: full absolute path (e.g., 'root/config/ZwjVersion') to the
        element's attribute dictionary. Paths are built from tag names.

    Notes
    -----
    - Attributes are recorded; element text is included as a tag element key.
    - The traversal is depth-first and includes every element exactly once.

    Examples
    --------
    >>> import xml.etree.ElementTree as ET
    >>> root = ET.fromstring('<a><b x="1"/><c y="2"/></a>')
    >>> parse_et_to_dict(root)
    {'a': {}, 'a/b': {'x': '1'}, 'a/c': {'y': '2'}}
    """
    elements = {}
    current_path = f"{element_path}/{element.tag}" if element_path else element.tag
    elements[current_path] = copy(element.attrib)

    if element.text and element.text.strip():
        if element.tag not in elements[current_path]:
            text_key = element.tag
        else:
            raise KeyError(f'Element tag "{element.tag}" already exist in attrib dict for "{current_path}".')
        elements[current_path].update({text_key: element.text})

    for child in element:
        elements.update(parse_et_to_dict(child, current_path))

    return elements

# all_elements = parse_et_to_dict(config)

In [ ]:


def get_et_element(path: str) -> ET.Element:
    """
    Create element from XML file.
    Parameters
    ----------
    path:str
    Path to XML file.

    Returns
    -------
    ET.Element object
    Root element from XML file.
    """
    with open(path, 'r', encoding='gb2312') as f:
        text = f.read()
        element = ET.fromstring(text)
    return element


element = get_et_element(xml_file)
all_elements = parse_et_to_dict(element)
settings = parse_et_to_dict(get_et_element(xml_settings))

In [ ]:
def search_dicts(dicts: dict[str, dict[str, str]], pattern: str, regex: bool = True, key: bool = True,
                 value: bool = True) -> dict:
    """
    Search dictionary for pattern matching.
    Parameters
    ----------
    dicts
    pattern
    regex
    key
    value

    Returns
    -------

    """
    result = {}

    if regex == False:
        for dictionary in dicts.values():
            if key:
                result.update({key: value for key, value in dictionary.items() if pattern in key})
            if value:
                result.update({key: value for key, value in dictionary.items() if pattern in value})
    else:
        pattern = re.compile(pattern)
        for dictionary in dicts.values():
            if key:
                result.update({key: value for key, value in dictionary.items() if pattern.match(key)})
            if value:
                result.update({key: value for key, value in dictionary.items() if pattern.match(value)})
    return result


print(search_dicts(all_elements, r'LK[\d]{2}'))


# Разработка репрезентации свойств ndax в классе.

## Объявление классов

In [4]:
class XmlHandler:
    def __init__(self, path: str):
        """

        Parameters
        ----------
        path
        """
        self.path = path
        self.element = self._get_el(self.path)
        self.dict = self._parse_et_to_dict(self.element)

    @staticmethod
    def _get_el(path: str) -> ET.Element:
        """
        Create element from XML file.
        Parameters
        ----------
        path:str
        Path to XML file.

        Returns
        -------
        ET.Element object
        Root element from XML file.
        """
        with open(path, 'r', encoding='gb2312') as f:
            text = f.read()
            element = ET.fromstring(text)
        return element

    def _parse_et_to_dict(self, element: ET.Element, element_path: str = '') -> dict[str, dict[str, str]]:
        """Build a mapping from full XML paths to element attribute dicts.

        Parameters
        ----------
        element : xml.etree.ElementTree.Element
            Root element to traverse.
        element_path : str, optional
            Prefix path to start from. Used internally during recursion.

        Returns
        -------
        dict[str, dict[str, str]]
            Mapping: full absolute path (e.g., 'root/config/ZwjVersion') to the
            element's attribute dictionary. Paths are built from tag names.

        Notes
        -----
        - Attributes are recorded; element text is included as a tag element key.
        - The traversal is depth-first and includes every element exactly once.

        Examples
        --------
        >>> import xml.etree.ElementTree as ET
        >>> root = ET.fromstring('<a><b x="1"/><c y="2"/></a>')
        >>> parse_et_to_dict(root)
        {'a': {}, 'a/b': {'x': '1'}, 'a/c': {'y': '2'}}
        """
        element_dict = {}
        current_path = f"{element_path}/{element.tag}" if element_path else element.tag
        element_dict[current_path] = copy(element.attrib)

        if element.text and element.text.strip():
            if element.tag not in element_dict[current_path]:
                text_key = element.tag
            else:
                raise KeyError(f'Element tag "{element.tag}" already exist in attrib dict for "{current_path}".')
            element_dict[current_path].update({text_key: element.text})

        for child in element:
            element_dict.update(self._parse_et_to_dict(child, current_path))

        return element_dict

    def search(self,
               pattern: str,
               regex: bool = True,
               key: bool = True,
               key2: bool = True,
               value: bool = True) -> dict:
        """
        Search dictionary for pattern matching.
        Parameters
        ----------
        pattern
        regex
        key
        value

        Returns
        -------

        """
        result = {}

        if regex == False:
            for dict_key, dictionary in self.dict.items():
                if key:
                    result.update(
                        {key_value: dict_value for key_value, dict_value in dictionary.items() if pattern in key_value})
                if value:
                    result.update({key_value: dict_value for key_value, dict_value in dictionary.items() if
                                   pattern in dict_value})
                if key2:
                    result.update(
                        {f"{dict_key}/{key_value}": dict_value for key_value, dict_value in dictionary.items() if
                         pattern in dict_key})
        else:
            pattern = re.compile(pattern)
            for dict_key, dictionary in self.dict.items():
                if key:
                    result.update(
                        {f"{dict_key}/{key_value}": dict_value for key_value, dict_value in dictionary.items() if
                         pattern.search(key_value)})
                if value:
                    result.update(
                        {f"{dict_key}/{key_value}": dict_value for key_value, dict_value in dictionary.items() if
                         pattern.search(dict_value)})
                if key2:
                    result.update(
                        {f"{dict_key}/{key_value}": dict_value for key_value, dict_value in dictionary.items() if
                         pattern.search(dict_key)})
        return result


In [5]:
class NdaxSettings:

    def __init__(self, path):
        self.path = path
        self.settings = {}
        self._parse_xml()

    def _parse_xml(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            zf = zipfile.PyZipFile(self.path)
            for xml_temp in zf.namelist():
                if '.xml' in xml_temp:
                    self.settings[xml_temp] = XmlHandler(zf.extract(xml_temp, path=tmpdir))

    def search(self, pattern: str, **kwargs) -> dict:
        result = {}
        for xml_name, xml in self.settings.items():
            temp_result = xml.search(pattern, **kwargs)
            result.update({f"{xml_name}/{key}": value for key, value in temp_result.items()})
        return result


In [6]:
def search_pattern(values, pattern):
    result = []
    if isinstance(values, str):
        r = pattern.search(values)
        if r:
            result.append(r.groups())

    else:
        for value in values:
            r = pattern.search(value)
            if r:
                result.append(r.groups())

    return result


def search_cell_code(settings, pattern):
    result = {}
    pattern = re.compile(pattern)
    result.update({'path': search_pattern(settings.path, pattern)})
    result.update({'barcode': search_pattern(settings.search('Barcode').values(), pattern)})
    result.update({'Remark': search_pattern(settings.search('Remark').values(), pattern)})

    return result


def flatten(obj):
    if isinstance(obj, dict):
        obj = obj.values()

    if isinstance(obj, (list, tuple, set, ValuesView)):
        for item in obj:
            yield from flatten(item)
    else:
        yield obj


def compare_repeats(iterable: list | tuple | ValuesView | str) -> list | None:
    iterable = list(set(list(flatten(iterable))))
    if len(iterable) == 0:
        return None
    elif len(iterable) == 1:
        return iterable[0]
    else:
        logger.log(2, 'Not consistent data')
        return iterable

In [7]:
class Check_pattern:  # TODO перенести в файл с обработкой (который будет)
    """
    Creates object that checks if given value list/dataframe corresponds to given filter_pattern.
    Pattern engine use slicing window with size equal to filter_pattern length. Each filter_pattern element is dict in form {'column_name':(compare operation, value)} for one condition per column or {'column_name':[(compare operation1, value1), (compare operation2, value2)]} for multiple conditions for the column.
    Args:
        pattern:
    """

    def __init__(self, pattern: list, ):
        self.length = len(pattern)
        self.pattern = [self.transform_pattern(step) for step in pattern]

    def __call__(self, split: pd.DataFrame):
        """
        Checks if values in list are consistent with filter_pattern. List length should be
        equal to filter_pattern length.
        Args:
            split (): list of values.

        Returns:

        """
        assert len(self.pattern) == len(split)
        checks = []
        for series, conditions in zip(split.reset_index(drop=True).iterrows(), self.pattern):
            check = self.check_series(series[1],
                                      conditions)  #iterrows return pairs of (index, series), so we choose series
            if not check:
                return False
            checks.append(check)
        return all(checks)

    def __len__(self):
        return len(self.pattern)

    @staticmethod
    def transform_pattern(step):
        if isinstance(step, list):
            return step
        step_transform = []
        for key in step.keys():
            if len(step[key]) > 1 and isinstance(step[key], list):
                step_transform.extend([(key, *comparator) for comparator in step[key]])
            else:
                step_transform.append((key, *step[key]))
        return step_transform

    def check_series(self, series, conditions):
        results = []
        for column, operator, value in conditions:
            if operator == 'exists':
                result = column in series.index
            elif column not in series.index:
                # print(f'No column {column}')
                result = True
            else:
                result = self.check_condition(a=series[column], b=value, operation=operator)
            if not result:
                return False
            results.append(result)
        return all(results)

    @staticmethod
    def check_condition(a, b, operation):
        match operation:
            case 'less':
                return a < b
            case 'more':
                return a > b
            case 'equal':
                return a == b
            case 'contain':
                return b in a
            case 'approx':
                borders = [b * 0.98, b * 1.02]
                borders.sort()
                return (a >= borders[0]) and (a <= borders[1])


def window_slider(window_size, stat_data):
    len_data = len(stat_data)
    for i in range(len_data - window_size + 1):
        yield i, i + window_size


def find_pattern(statistics: pd.DataFrame, pattern: list):
    pattern_checker = Check_pattern(pattern)
    window_size = len(pattern_checker)

    slices = window_slider(window_size, statistics)
    roll = {statistics.iloc[a].name: True for a, b in slices if pattern_checker(statistics.iloc[a:b])}
    roll = pd.DataFrame.from_dict(roll, 'index', columns=['result'])
    windows = []
    for i in roll[roll['result']].index:
        start_index = i
        end_index = start_index + window_size
        windows.append(list(range(start_index, end_index)))
    return windows


def extract_pattern(data, pattern):
    slices = []
    windows = find_pattern(data, pattern)
    for window in windows:
        slices.append(data.loc[window])
    return slices


In [8]:
#pattern creation
#operations: 'contain' 'equal' 'more' 'less' 'approx' 'exists'
pattern_cycle_full = [
    {'Status_unique_values': ('contain', '_Chg'), 'Voltage_last': ('approx', 4), 'T1_max': ('less', 54.9)},
    {'Status_unique_values': ('contain', 'Rest')},
    {'Status_unique_values': ('contain', '_DChg'), 'Voltage_last': ('approx', 1.5), 'T1_max': ('less', 54.9)},
    {'Status_unique_values': ('contain', 'Rest')}
]
pattern_cycle_low = [
    {'Status_unique_values': ('contain', '_Chg'), 'Voltage_last': ('approx', 3.8), 'T1_max': ('less', 54.9)},
    {'Status_unique_values': ('contain', 'Rest')},
    {'Status_unique_values': ('contain', '_DChg'), 'Voltage_last': ('approx', 1.5), 'T1_max': ('less', 54.9)},
    {'Status_unique_values': ('contain', 'Rest')}
]
pattern_cycle_t1 = [
    {'Status_unique_values': ('contain', '_Chg'), 'T1_max': [('exists', 1), ('more', 54.9)]},
    {'Status_unique_values': ('contain', 'Rest')},
    {'Status_unique_values': ('contain', '_DChg'), 'Voltage_last': ('approx', 1.5), 'T1_max': ('less', 54.9)},
    {'Status_unique_values': ('contain', 'Rest')}
]
pattern_cycle_t2 = [
    {'Status_unique_values': ('contain', '_Chg'), 'Voltage_last': ('more', 3.79), 'T1_max': ('less', 54.9)},
    {'Status_unique_values': ('contain', 'Rest')},
    {'Status_unique_values': ('contain', '_DChg'), 'T1_max': [('exists', 1), ('more', 54.9)]},
    {'Status_unique_values': ('contain', 'Rest')}
]
pattern_resistance = [
    {'Status_unique_values': ('contain', 'Rest'), 'Time_range': ('approx', 1800)},
    {'Status_unique_values': ('contain', '_DChg'), 'Time_range': ('approx', 30)},
    {'Status_unique_values': ('contain', '_DChg'), 'Time_range': ('approx', 5)},
    {'Status_unique_values': ('contain', 'Rest'), 'Time_range': ('approx', 1800)}
]

## Разработка

In [ ]:
# pattern = r'EndTime'
pattern = r'(S[A,B,А,В]-?\d{2})'
data_path = NdaxSettings(sa_ndax)
print(data_path.search(pattern))

### Парсинг обозначений ячеек

Тесты отдельных путей

In [ ]:
#тест общей функции импорта
files = r'D:\!Science\Analysis\Electrochem\2024 Na-ion\2024-10-29 Определение параметров\SA12(сработала защита)\Opredelenie parametrov SA 12-BTS83-102-2-8-85.ndax'
setting = NdaxSettings(files)
cell_code = search_cell_code(setting, pattern)
flat_code = compare_repeats(cell_code.values())
# print(files, flat_code)

start_date = setting.search(start_date_pattern)
start_date = compare_repeats(start_date.values())
end_date = setting.search(end_date_pattern)
print(end_date)
end_date = compare_repeats(end_date.values())
cur_date = setting.search('date')
# print(cur_date)
cur_date = cur_date['VersionInfo.xml/root/config/date']
cur_date = datetime.strptime(cur_date, "%Y%m%d%H%M%S")
if flat_code is None:
    pass
elif not isinstance(flat_code, str):
    flat_code = {dict_key: compare_repeats(dict_value) for dict_key, dict_value in cell_code.items() if
                 compare_repeats(dict_value)}
    flat_code = flat_code['path']
else:
    flat_code = flat_code.replace('-', '')
# print((files, flat_code, start_date, end_date, cur_date))
print(end_date)

Парсинг всего массива данных

In [ ]:
directory = r'D:\!Science\Analysis\Electrochem\2024 Na-ion'
pattern = r'(S[A,B,А,В]-?\d{2})'
start_date_pattern = 'StartTime'
end_date_pattern = 'EndTime'
cur_date_pattern = 'date'
list_files = battery_parser.list_files(directory, 'ndax')
tuples = []
for files in list_files:
    setting = NdaxSettings(files)
    cell_code = search_cell_code(setting, pattern)
    flat_code = compare_repeats(cell_code.values())
    # print(files, flat_code)

    start_date = setting.search(start_date_pattern)
    start_date = compare_repeats(start_date.values())
    end_date = setting.search(end_date_pattern)
    end_date = compare_repeats(end_date.values())
    cur_date = setting.search('date')
    cur_date = cur_date['VersionInfo.xml/root/config/date']
    cur_date = datetime.strptime(cur_date, "%Y%m%d%H%M%S")
    if not end_date:
        end_date = cur_date
    if flat_code is None:
        pass
    elif not isinstance(flat_code, str):
        flat_code = {dict_key: compare_repeats(dict_value) for dict_key, dict_value in cell_code.items() if
                     compare_repeats(dict_value)}
        flat_code = flat_code['path']
    else:
        flat_code = flat_code.replace('-', '')

    tuples.append((files, flat_code, start_date, end_date, cur_date))
data = pd.DataFrame(tuples, columns=['filepath', 'cell_code', 'start_date', 'end_date', 'cur_date'])


In [ ]:
#format data dataframe
data.rename(columns={"filepath": "path"}, inplace=True)
data = data.astype(
    {'cell_code': 'string', 'path': 'string', 'start_date': 'datetime64[ns]', 'end_date': 'datetime64[ns]'}, copy=False)

In [ ]:
#test save-load - success
info_filepath = directory + '\\' + 'cell_info.parquet'
data.to_parquet(info_filepath, )
data_load = pd.read_parquet(info_filepath)
data.equals(data_load)

In [ ]:
# check null values number

for i in data[data['end_date'].isnull()]['path']:
    print(i)

Всего 244 записи без end_date

In [ ]:
#create unique cell-stepinfo list
step_info_list = []
settings = {}
for files in list_files:
    settings[files] = NdaxSettings(files)

for files in list_files:
    setting = settings[files]
    step_info = setting.search('Step_Info')
    step_info['cell_code'] = data[data['path'] == files]['cell_code'].item()
    step_info['start_date'] = data[data['path'] == files]['start_date'].item()
    if step_info not in step_info_list:
        step_info_list.append(step_info)

fingerprints = {}
for files in list_files:
    setting = settings[files]
    step_info = setting.search('Step_Info')
    step_info['cell_code'] = data[data['path'] == files]['cell_code'].item()
    step_info['start_date'] = data[data['path'] == files]['start_date'].item()
    fingerprints[files] = step_info

In [ ]:
unique = []
for fingerprint in step_info_list:
    filtered = []
    for files in list_files:
        setting = settings[files]
        if fingerprints[files] == fingerprint:
            filtered.append(setting)
    if len(filtered) > 1:
        print('Comparison')
        for filt in filtered:
            print(filt.path)
            print(data[data['path'] == filt.path]['end_date'].item())
    maximum_date = max(filtered, key=lambda x: data[data['path'] == x.path]['end_date'].item())
    if len(filtered) > 1:
        print('Winner: ', maximum_date.path)
    unique.append(maximum_date)

In [ ]:
unique_paths = [i.path for i in unique]
filtered_data = data[data['path'].isin(unique_paths)]

# Работа с фильтрованными данными

## Построение статистики по фильтрованным данным

In [ ]:
#Сохранение фильтрованных данных

# filtered_data.to_parquet(filtered_datapath)
filtered_data = pd.read_parquet(filtered_datapath)

In [ ]:
#test step index saving
filepath = 'D:\\!Science\\Analysis\\Electrochem\\2024 Na-ion\\2025-01-10 target SoH cycling\\Данные 2025-05-30\\102-1-6-SB46.ndax'
test_data = NewareNDA.read(filepath)

In [ ]:
#test parsing
filepath = 'D:\\!Science\\Analysis\\Electrochem\\2024 Na-ion\\2025-01-10 target SoH cycling\\Данные 2025-05-30\\102-1-6-SB46.ndax'
test_data = NewareNDA.read(filepath)

statistic_pattern = {
    'Status': 'unique_values',
    'Step_Index': 'unique_values',
    'Step': 'unique_values',
    'Index': ['first', 'last'],
    'Current(mA)': ['first', 'last', 'mean', 'median', 'std'],
    'Time': ['range', 'diff'],
    'Charge_Capacity(mAh)': 'max',
    'Discharge_Capacity(mAh)': 'max',
    'Voltage': ['first', 'last', 'mean'],
    'Timestamp': ['min', 'max']
}
if 'T1' in test_data.columns:
    statistic_pattern['T1'] = ['first', 'last', 'mean', 'min', 'max']
test_data = test_data.astype({"Step_Index": "string"})
battery_parser.statistics.generate_statistics(test_data, statistics_pattern=statistic_pattern)

In [ ]:
#full parsing to statistics
statistics = {}
statistics_directory = directory + '\\' + 'statistics'
statistics_path_df = {'path': [], 'statistic_path': []}
for i, data_path in enumerate(filtered_data['path']):
    test_data = NewareNDA.read(data_path)
    statistic_pattern = {
        'Status': 'unique_values',
        'Step_Index': 'unique_values',
        'Step': 'unique_values',
        'Index': ['first', 'last'],
        'Current(mA)': ['first', 'last', 'mean', 'median', 'std'],
        'Time': ['range', 'diff'],
        'Charge_Capacity(mAh)': 'max',
        'Discharge_Capacity(mAh)': 'max',
        'Voltage': ['first', 'last', 'mean'],
        'Timestamp': ['min', 'max']}
    if 'T1' in test_data.columns:
        statistic_pattern['T1'] = ['first', 'last', 'mean', 'min', 'max']
    test_data = test_data.astype({"Step_Index": "string"})
    test_data = test_data.astype({"Step": "string"})
    statistics[data_path] = battery_parser.statistics.generate_statistics(test_data,
                                                                          statistics_pattern=statistic_pattern)
    statistics_path_df['path'].append(data_path)
    statistics_path = statistics_directory + '\\' + f'{i}.parquet'
    statistics_path_df['statistic_path'].append(statistics_path)
    statistics[data_path].to_parquet(statistics_path)


In [ ]:
#extend info dataframe with paths to statistics (only once!)
# statistics_df = pd.DataFrame(statistics_path_df, columns=['path', 'statistic_path'])
# filtered_data = filtered_data.merge(statistics_df, on='path', how='left')


In [ ]:
#save global info dataframe
filtered_data.to_parquet(filtered_datapath)

In [ ]:
#save statistics dataframes
filtered_data = pd.read_parquet(filtered_datapath)
for datapath, statistic_path in zip(filtered_data['path'], filtered_data['statistic_path']):
    statistics[datapath].index = statistics[datapath].index.astype('int')
    statistics[datapath].sort_index(inplace=True)
    statistics[datapath].to_parquet(statistic_path)

# Работа со статистикой

## Организация среды, загрузка данных статистики

In [9]:
# load statistics dataframes
filtered_data = pd.read_parquet(filtered_datapath)
statistics = {}
for datapath, statistic_path in zip(filtered_data['path'], filtered_data['statistic_path']):
    statistics[datapath] = pd.read_parquet(statistic_path)


## Дополнить статистику общей  ёмкостью

In [ ]:
#test methods

test_df = statistics[filtered_data['path'].iloc[40]]
filter_Q = test_df[['Charge_Capacity(mAh)_max', 'Discharge_Capacity(mAh)_max']]
filter_Q = filter_Q.loc[(filter_Q > 0).all(axis=1)]
print(filter_Q.size)

In [ ]:
#test if there are charge and discharge capacity at the same time

for key, value in statistics.items():
    filter_Q = value[['Charge_Capacity(mAh)_max', 'Discharge_Capacity(mAh)_max']]
    filter_Q = filter_Q.loc[(filter_Q > 0).all(axis=1)]
    if filter_Q.size > 0:
        print('Unmerging capacity:', key)

Всё ок, ёмкость либо зарядная либо разрядная. Можно объединить колонку с ёмкостью.

In [ ]:
#create new statistics column with capacity (only once!)
for key, value in statistics.items():
    filter_Q = value[['Charge_Capacity(mAh)_max', 'Discharge_Capacity(mAh)_max']]
    value['Capacity(mAh)_max'] = filter_Q.sum(axis=1)
    statistics[key] = value

In [ ]:
#check if there are new column in statistics
for key, value in statistics.items():
    print(value['Capacity(mAh)_max'].size)

## Разработка парсера полных циклов

In [ ]:
#create selected view for dataframes
# 'Status_unique_values', 'Current(mA)_first','Time_range', 'Voltage_last', 'T1_max', 'Capacity(mAh)_max'
view = ['Status_unique_values',
        'Current(mA)_first',
        'Voltage_last',
        'T1_max',
        'Capacity(mAh)_max']

In [ ]:
#test data creation
data_path = filtered_data['path'].iloc[1]
test_df = statistics[data_path]
print(test_df.shape)
#test search dataframe
result = extract_pattern(test_df, pattern_cycle_full)

In [ ]:
for key, statistic in statistics.items():

    a = statistic[statistic['Status_unique_values'].isin(['CCCV_Chg', 'CC_Chg', 'CC_DChg', 'CCCV_DChg'])]
    a = a[['Status_unique_values', 'Current(mA)_first', 'Current(mA)_median']]
    b = a['Current(mA)_first'] / a['Current(mA)_median']

    if not b[(b < 0.99) | (b > 1.01)].empty:
        print(a[(b < 0.99) | (b > 1.01)])

## Парсинг и запись экспериментов

In [16]:
analysis_df = pd.DataFrame(
    columns=['cell_code', 'test_type', 'index_start', 'index_finish', 'Ch_cap', 'DCh_cap', 'Ch_curr', 'DCh_curr',
             'Ch_Tmax', 'DCh_Tmax', 'E0', 'R', 'timestamp', 'path'])


In [17]:
#create function for cycle pattern creation
def create_cycle_info(df, cell_code, test_type, path):
    view = ['Capacity(mAh)_max', 'Current(mA)_first', 'Current(mA)_median', 'Timestamp_min']
    if 'T1_max' in df.columns:
        view.append('T1_max')
    df_view = df[view]
    new_row = {'cell_code': cell_code,
               'test_type': test_type,
               'index_first': df_view.index[0],
               'index_last': df_view.index[-1],
               'Ch_cap': df_view.iloc[0, 0],
               'DCh_cap': df_view.iloc[2, 0],
               'Ch_curr': select_current(df_view.iloc[0, 1], df_view.iloc[0, 2]),
               'DCh_curr': select_current(df_view.iloc[2, 1], df_view.iloc[2, 2]),
               'timestamp': df_view.iloc[0, 3],
               'path': path}
    if 'T1_max' in df.columns:
        new_row.update({'Ch_Tmax': df_view.iloc[0, 4],
                        'DCh_Tmax': df_view.iloc[2, 4]})
    return new_row


def select_current(cur_first, cur_median):
    if abs(cur_first) < abs(cur_median):
        return cur_median
    else:
        return cur_first

In [ ]:
# parse full cycles
total = len(statistics)
for key, statistic in statistics.items():
    data_list = extract_pattern(statistic, pattern_cycle_full)
    cell_code = filtered_data[filtered_data['path'] == key]['cell_code'].item()
    path = key
    test_type = '4V_cycle'
    for df in data_list:
        new_row = create_cycle_info(df, cell_code, test_type, path)
        analysis_df.loc[len(analysis_df)] = pd.Series(new_row)


In [ ]:
# parse full cycles low voltage
for key, statistic in statistics.items():
    data_list = extract_pattern(statistic, pattern_cycle_low)
    cell_code = filtered_data[filtered_data['path'] == key]['cell_code'].item()
    path = key
    test_type = '3.8V_cycle'
    # print(len(data_list))
    for df in data_list:
        new_row = create_cycle_info(df, cell_code, test_type, path)
        analysis_df.loc[len(analysis_df)] = pd.Series(new_row)

In [ ]:
# parse overheat charge cycles
for key, statistic in statistics.items():
    data_list = extract_pattern(statistic, pattern_cycle_t1)
    cell_code = filtered_data[filtered_data['path'] == key]['cell_code'].item()
    path = key
    test_type = 'overheat_cycle_charge'
    # print(len(data_list))
    for df in data_list:
        new_row = create_cycle_info(df, cell_code, test_type, path)
        analysis_df.loc[len(analysis_df)] = pd.Series(new_row)

In [ ]:
# parse overheat discharge cycles
for key, statistic in statistics.items():
    data_list = extract_pattern(statistic, pattern_cycle_t2)
    cell_code = filtered_data[filtered_data['path'] == key]['cell_code'].item()
    path = key
    test_type = 'overheat_cycle_discharge'
    # print(len(data_list))
    for df in data_list:
        new_row = create_cycle_info(df)
        analysis_df.loc[len(analysis_df)] = pd.Series(new_row)

In [ ]:
# parse resistance
for key, statistic in statistics.items():
    data_list = extract_pattern(statistic, pattern_resistance)
    cell_code = filtered_data[filtered_data['path'] == key]['cell_code'].item()
    path = key
    test_type = 'resistance'
    # print(len(data_list))
    for df in data_list:
        view = ['Current(mA)_median', 'Voltage_last', 'Timestamp_min']
        if 'T1_max' in df.columns:
            view.append('T1_max')
        df_view = df[view]
        resistance = (df_view.iloc[2, 1] - df_view.iloc[1, 1]) / (df_view.iloc[2, 0] - df_view.iloc[1, 0]) * 1000
        new_row = {'cell_code': cell_code,
                   'test_type': test_type,
                   'index_first': df_view.index[0],
                   'index_last': df_view.index[-1],
                   'E0': df_view.iloc[0, 1],
                   'R': resistance,
                   'timestamp': df_view.iloc[0, 2],
                   'path': path}
        if 'T1_max' in df.columns:
            new_row.update({'Ch_Tmax': df_view.iloc[0, 3],
                            'DCh_Tmax': df_view.iloc[2, 3]})
        analysis_df.loc[len(analysis_df)] = pd.Series(new_row)

In [ ]:
#save data
analysis_df.to_parquet(step_datapath)

## Обобщение полученных данных

In [ ]:
analysis_df = pd.read_parquet(step_datapath)

In [ ]:
analysis_df[analysis_df['DCh_curr'] > -130]

In [ ]:
statistics[
    'D:\\!Science\\Analysis\\Electrochem\\2024 Na-ion\\2024-09-10 Состаривание 1\\3. Определение параметров\\SA26\\250102-3-4-79.ndax']

В каждом файле xml содержится обычно по 1 дате.

в EditStep содержится время запуска, в Step, TestInfo и VersionInfo - время окончания.



# Задачи для разработки
-Подобрать набор небольших файлов .ndax для тестовой обработки. Необходимо разнообразие
    - Для испытания xml - с barcode и remarks, без одного из них,

    - представлены различные техники испытаний
    - разные силы тока

